# BirdCLEF+ 2026 - Perch v2 Probe Submission

Purpose: load Google Perch v2, extract test soundscape embeddings, apply the trained PyTorch probe, and write `submission.csv` for Kaggle.

Artifacts are written to `/kaggle/working/artifacts/perch_submission`.


## 1. Setup


In [2]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
        Path("data/raw/birdclef-2026"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path) if path.exists() else None


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Artifacts: {CFG.artifact_dir}")

import zipfile
import subprocess
from importlib.metadata import PackageNotFoundError, version

import librosa
import torch
from torch import nn
from tqdm.auto import tqdm
from IPython.display import FileLink

REQUIRED_TENSORFLOW_VERSION = "2.20.0"
OFFLINE_TENSORFLOW_WHEELHOUSE = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel")


def package_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple(REQUIRED_TENSORFLOW_VERSION):
    import sys
    print(f"TensorFlow {installed_tf} is not compatible with Perch v2 export 2.")
    print(f"Installing tensorflow=={REQUIRED_TENSORFLOW_VERSION}. Restart the Kaggle session after this cell stops.")
    if OFFLINE_TENSORFLOW_WHEELHOUSE.exists():
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(OFFLINE_TENSORFLOW_WHEELHOUSE / "tensorboard-2.20.0-py3-none-any.whl"),
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(
                    OFFLINE_TENSORFLOW_WHEELHOUSE
                    / "tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
                ),
            ],
            check=True,
        )
        print("Installed TF 2.20.0 from Kaggle dataset wheels.")
    else:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.20.0", "tensorboard==2.20.0"],
            check=True,
        )
        print("Installed TF 2.20.0 from PyPI.")
    installed_tf = package_version("tensorflow")
    if installed_tf is None or version_tuple(installed_tf) < version_tuple(REQUIRED_TENSORFLOW_VERSION):
        raise RuntimeError(
            "TensorFlow upgrade did not complete. If Kaggle shows DNS or connection errors, enable Internet "
            "for this notebook or attach an offline TensorFlow wheelhouse dataset and set "
            "OFFLINE_TENSORFLOW_WHEELHOUSE to that /kaggle/input path."
        )
    raise SystemExit("TensorFlow was upgraded. Restart the Kaggle session, then run the notebook from the top.")

import tensorflow as tf

torch.set_num_threads(min(2, os.cpu_count() or 1))


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/perch_submission")
    sample_rate = 32000
    duration = 5.0
    extraction_batch_size = 8
    probe_batch_size = 256
    hidden_dim = 512
    dropout = 0.25
    perch_model_dir = None
    probe_checkpoint_path = None
    labels_path = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"TensorFlow: {tf.__version__}")


Data root: /kaggle/input/competitions/birdclef-2026
Artifacts: /kaggle/working/artifacts
Device: cuda
TensorFlow: 2.20.0


## 2. Locate Competition Files And Model Artifacts


In [3]:
def find_file(filename: str, roots: list[Path]) -> Path:
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        matches = list(root.glob(f"**/{filename}"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Attach the Perch probe model artifact.")


def find_perch_model_dir() -> Path:
    if CFG.perch_model_dir:
        return Path(CFG.perch_model_dir)
    input_root = Path("/kaggle/input")
    matches = list(input_root.glob("**/saved_model.pb")) if input_root.exists() else []
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError("Could not find Google Perch v2 SavedModel. Attach the Kaggle Perch model.")


artifact_roots = [Path("/kaggle/input"), Path("artifacts/perch_v2")]
sample_path = CFG.data_root / "sample_submission.csv"
probe_checkpoint_path = Path(CFG.probe_checkpoint_path) if CFG.probe_checkpoint_path else find_file("best_perch_probe.pt", artifact_roots)
labels_path = Path(CFG.labels_path) if CFG.labels_path else find_file("labels.json", artifact_roots)
perch_model_dir = find_perch_model_dir()

sample_submission = pd.read_csv(sample_path)
idx_to_label = {int(k): v for k, v in json.loads(labels_path.read_text()).items()}
labels = [idx_to_label[i] for i in sorted(idx_to_label)]
label_to_idx = {label: i for i, label in enumerate(labels)}
target_columns = [c for c in sample_submission.columns if c != "row_id"]

print(f"Sample submission: {sample_path}")
print(f"Perch model: {perch_model_dir}")
print(f"Probe checkpoint: {probe_checkpoint_path}")
print(f"Labels: {labels_path}")
print(f"Rows: {len(sample_submission):,}")
print(f"Submission target columns: {len(target_columns):,}")
print(f"Probe classes: {len(labels):,}")
display(sample_submission.head())


Sample submission: /kaggle/input/competitions/birdclef-2026/sample_submission.csv
Perch model: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
Probe checkpoint: /kaggle/input/models/tuannm3823/birdclef-2026-perch-v2-probe/pytorch/baseline-v1/1/best_perch_probe.pt
Labels: /kaggle/input/models/tuannm3823/birdclef-2026-perch-v2-probe/pytorch/baseline-v1/1/labels.json
Rows: 3
Submission target columns: 234
Probe classes: 206


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## 3. Load Perch And Probe


In [4]:
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


def explain_perch_runtime_error(error: Exception) -> None:
    message = str(error)
    if "XlaCallModuleOp with version 10 is not supported" in message:
        raise RuntimeError(
            "This Perch SavedModel requires TensorFlow/XLA >= the version installed in this session. "
            "The setup cell should install tensorflow>=2.20 before import. Restart the Kaggle session "
            "after installation, then run from the top."
        ) from error
    raise error


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: tensor})
        else:
            outputs = infer(tensor)
    except Exception as error:
        explain_perch_runtime_error(error)

    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (name for name in arrays if "embedding" in name.lower() and "spatial" not in name.lower()),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


dummy = np.zeros((1, int(CFG.sample_rate * CFG.duration)), dtype=np.float32)
dummy_embedding = run_perch_batch(dummy)
print(f"Smoke-test embedding shape: {dummy_embedding.shape}")


class PerchProbe(nn.Module):
    def __init__(self, embedding_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def torch_load(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = torch_load(probe_checkpoint_path)
probe = PerchProbe(embedding_dim=dummy_embedding.shape[1], num_classes=len(labels)).to(device)
probe.load_state_dict(checkpoint["model"])
probe.eval()


I0000 00:00:1778333099.082599      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778333099.085534      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Inputs: ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
Outputs: {'embedding': TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding'), 'spatial_embedding': TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding'), 'spectrogram': TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram'), 'label': TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')}


2026-05-09 13:25:09.218719: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-09 13:25:09.360877: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-09 13:25:10.736376: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-09 13:25:10.875487: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-09 13:25:11.061636: E external/local_xla/xla/stream_

Smoke-test embedding shape: (1, 1536)


PerchProbe(
  (net): Sequential(
    (0): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=1536, out_features=512, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=512, out_features=206, bias=True)
  )
)

## 4. Build Test Audio Index


In [5]:
def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
    stem, sep, end = str(row_id).rpartition("_")
    if sep and end.replace(".", "", 1).isdigit():
        return stem, float(end)
    return str(row_id), CFG.duration


def build_audio_index() -> dict[str, Path]:
    candidates = [
        CFG.data_root / "test_soundscapes",
        CFG.data_root / "test_audio",
        CFG.data_root / "train_soundscapes",
    ]
    audio_index = {}
    for folder in candidates:
        if folder.exists():
            for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                for path in folder.glob(ext):
                    audio_index[path.stem] = path
    return audio_index


def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


audio_index = build_audio_index()
print(f"Indexed audio files: {len(audio_index):,}")


Indexed audio files: 10,658


## 5. Run Perch Probe Inference


In [6]:
def predict_probe(embeddings: np.ndarray) -> np.ndarray:
    x = torch.from_numpy(embeddings.astype(np.float32)).to(device)
    with torch.no_grad():
        logits = probe(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs


submission = sample_submission.copy()
for col in target_columns:
    submission[col] = 0.0

waveforms = []
batch_rows = []
missing_audio = []

for row_idx, row_id in tqdm(list(enumerate(submission["row_id"])), desc="perch submission"):
    stem, end_time = row_to_stem_and_end_time(row_id)
    audio_path = audio_index.get(stem)
    if audio_path is None:
        missing_audio.append(row_id)
        continue

    waveforms.append(load_audio_segment(audio_path, end_time))
    batch_rows.append(row_idx)

    if len(waveforms) == CFG.extraction_batch_size:
        embeddings = run_perch_batch(np.stack(waveforms))
        probs = predict_probe(embeddings)
        for local_idx, submit_idx in enumerate(batch_rows):
            for label, class_idx in label_to_idx.items():
                if label in target_columns:
                    submission.loc[submit_idx, label] = probs[local_idx, class_idx]
        waveforms = []
        batch_rows = []

if waveforms:
    embeddings = run_perch_batch(np.stack(waveforms))
    probs = predict_probe(embeddings)
    for local_idx, submit_idx in enumerate(batch_rows):
        for label, class_idx in label_to_idx.items():
            if label in target_columns:
                submission.loc[submit_idx, label] = probs[local_idx, class_idx]

missing_audio_path = CFG.artifact_dir / "missing_test_audio.json"
missing_audio_path.write_text(json.dumps(missing_audio, indent=2), encoding="utf-8")
print(f"Missing audio rows: {len(missing_audio):,}")
display(submission.head())


perch submission:   0%|          | 0/3 [00:00<?, ?it/s]

Missing audio rows: 3


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BC2026_Test_0001_S05_20250227_010002_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BC2026_Test_0001_S05_20250227_010002_15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Save Submission And Package Artifacts


In [7]:
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(CFG.artifact_dir / "submission.csv", index=False)

zip_path = Path("/kaggle/working/birdclef_perch_v2_submission_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_path, arcname="submission.csv")
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote submission: {submission_path}")
print(f"Wrote artifact zip: {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(submission_path))
display(FileLink(zip_path))


Wrote submission: /kaggle/working/submission.csv
Wrote artifact zip: /kaggle/working/birdclef_perch_v2_submission_artifacts.zip (0.00 MB)


/kaggle/working/submission.csv

/kaggle/working/birdclef_perch_v2_submission_artifacts.zip